# Trabalho 1 - Recuperação da Informação

Grupo:

- Arthur Trottmann Ramos (14681052)
- Maicon Chaves Marques (14593530)

## Instalação de Dependências e Carregamento de Dataset

In [9]:
pip install NLTK numpy ir_datasets

Note: you may need to restart the kernel to use updated packages.


In [10]:
import ir_datasets

dataset = ir_datasets.load("cranfield")

## Pré-Processamento

In [11]:
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stemmer = PorterStemmer()


def tokenization(text):
  tokenizer = RegexpTokenizer(r'\w+')
  clean_tokens = tokenizer.tokenize(text)
  return lower_case_normalization(clean_tokens)

def remove_stopwords(words):
  stopwords_set = set(stopwords.words('english'))
  filtered_words = [word for word in words if word not in stopwords_set]
  return filtered_words

def lower_case_normalization(words):
  normalized_words = [word.lower() for word in words]
  return normalized_words

def stemming(words):
  stemmed_words = [stemmer.stem(word) for word in words]
  return stemmed_words

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Arthur\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
def preprocess(text, config_type=0):
  words = tokenization(text)

  if config_type == 1:
    words = remove_stopwords(words)
  if config_type == 2:
    words = stemming(words)
  if config_type == 3:
    words = remove_stopwords(words)
    words = stemming(words)

  return words

## Índice Invertido

In [13]:
class InvertedIndex:
  def __init__(self):
    self.index = {}
    self.document_length = {}
    self.n_documents = 0
    self.avgdl = 0.0

  def build(self, documents, preprocessing_type=0):
    for doc in documents:
      doc_id = doc[0]
      doc_title = doc[1]
      doc_text = doc[2]
      doc_author = doc[3]

      self.n_documents += 1

      title_words = preprocess(doc_title, preprocessing_type)
      text_words = preprocess(doc_text, preprocessing_type)
      author_words = preprocess(doc_author, preprocessing_type)

      self.document_length[doc_id] = len(title_words) + len(text_words) + len(author_words)

      for word in title_words:
        if word not in self.index:
          self.index[word] = {}
        if doc_id not in self.index[word]:
          self.index[word][doc_id] = 0
        self.index[word][doc_id] += 1

    self.avgdl = sum(self.document_length.values()) / len(self.document_length)

## Modelo Probabilístico (BM25)

In [14]:
import math

class BM25:
    def __init__(self, inverted_index, k1=1.5, b=0.75):
        self.inverted_index = inverted_index
        self.k1 = k1
        self.b = b
    
    def score(self, query, doc_id, preprocessing_type=0):
        score = 0.0
        query_words = preprocess(query, preprocessing_type)
    
        for word in query_words:
            if word in self.inverted_index.index and doc_id in self.inverted_index.index[word]:
                tf = self.inverted_index.index[word][doc_id]
                df = len(self.inverted_index.index[word])
                idf = math.log(1 + ((self.inverted_index.n_documents - df + 0.5) / (df + 0.5)))
                dl = self.inverted_index.document_length[doc_id]
                avgdl = self.inverted_index.avgdl
                score += idf * ((tf * (self.k1 + 1)) / (tf + self.k1 * (1 - self.b + self.b * (dl / avgdl))))
    
        return score

## Modelo Vetorial